In [15]:
#ô code 1 
import pandas as pd


df = pd.read_csv('data/Dataset-Unicauca-Version2-87Atts.csv')
df = df.sample(100000, random_state=42).reset_index(drop=True) #lấy 100k dòng chạy thử và đánh lại số tt chạy từ đầu 
# Lệnh này sẽ gom nhóm và đếm tất cả các loại app có trong file
print(df['ProtocolName'].unique())
print(f"Tổng số ứng dụng khác nhau là: {df['ProtocolName'].nunique()}")



<StringArray>
[          'HTTP',         'GOOGLE',     'HTTP_PROXY',   'HTTP_CONNECT',
      'MICROSOFT',         'AMAZON',  'CONTENT_FLASH',        'YOUTUBE',
          'GMAIL',            'SSL', 'WINDOWS_UPDATE',       'FACEBOOK',
            'MSN',          'SKYPE',        'DROPBOX',     'OFFICE_365',
        'TWITTER',     'CLOUDFLARE',     'TEAMVIEWER',          'YAHOO',
        'NETFLIX',        'IP_ICMP',            'DNS',        'SPOTIFY',
   'MS_ONE_DRIVE',       'WHATSAPP',          'APPLE',      'UBUNTUONE',
   'APPLE_ITUNES',    'GOOGLE_MAPS',           'EBAY',    'SSL_NO_CERT',
      'INSTAGRAM',           'MQTT',         'TIMMEU',       'FTP_DATA',
           'WAZE',           'H323',      'WIKIPEDIA',   'APPLE_ICLOUD',
       'EASYTAXI',         'RADIUS',        'EDONKEY',  'HTTP_DOWNLOAD',
         'DEEZER',            'TOR',            'NTP',    'FTP_CONTROL',
      'WHOIS_DAS',          'OSCAR',       'TELEGRAM',            'SSH',
         'ORACLE',            'BGP']


In [16]:
#ô code 2
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
import numpy as np

# 1. Mã hóa nhãn: Biến 'YOUTUBE', 'FACEBOOK'... thành số 0, 1, 2...
le = LabelEncoder()
df['ProtocolName_Encoded'] = le.fit_transform(df['ProtocolName'])

# 2. Chọn các cột đặc trưng (Features) - Loại bỏ các cột không phải số hoặc không cần thiết
# Mình tạm loại bỏ các cột IP và ID vì chúng dễ làm AI bị "học vẹt"
features = df.select_dtypes(include=[np.number]).columns.tolist()
features = [f for f in features if f not in ['ProtocolName_Encoded', 'L7Protocol']]

X = df[features]
y = df['ProtocolName_Encoded']

# 3. Xử lý giá trị vô hạn (Inf) hoặc lỗi dữ liệu thường gặp trong lưu lượng mạng
X = X.replace([np.inf, -np.inf], np.nan).fillna(0)

# 4. Chia dữ liệu: 80% để học, 20% để kiểm tra
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 5. Chuẩn hóa: Đưa các con số về cùng một thang đo (0-1 hoặc tương đương)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Đã chuẩn bị xong dữ liệu cho {len(features)} đặc trưng và {df['ProtocolName'].nunique()} ứng dụng.")


Đã chuẩn bị xong dữ liệu cho 80 đặc trưng và 54 ứng dụng.


In [17]:
#ô code 3 : mã hóa nhãn
from sklearn.preprocessing import LabelEncoder

#  Tái mã hóa nhãn riêng cho tập Train và Test để đảm bảo tính liên tục (0, 1, 2...)
# Bước này cực kỳ quan trọng để sửa lỗi "Invalid classes inferred"
le_final = LabelEncoder()
y_train_fixed = le_final.fit_transform(y_train)

# Đối với tập Test, chúng ta chỉ lấy những nhãn mà tập Train đã học được
# Những nhãn nào tập Train không có sẽ bị loại bỏ ở tập Test để không gây lỗi
test_mask = y_test.isin(le_final.classes_)
X_test_fixed = X_test_scaled[test_mask]
y_test_fixed = le_final.transform(y_test[test_mask])

In [18]:
#ô code 4
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, accuracy_score

# 1. Khởi tạo mô hình XGBoost
model_baseline = XGBClassifier(n_estimators=100, random_state=42, n_jobs=-1)

# 2. Huấn luyện mô hình
print(f"Đang huấn luyện mô hình cơ sở trên {len(le_final.classes_)} ứng dụng...")
model_baseline.fit(X_train_scaled, y_train_fixed)

# 3. Dự đoán và đánh giá
y_pred = model_baseline.predict(X_test_fixed)
print(f"\n--- KẾT QUẢ HOÀN THÀNH WEEK 3 ---")
print(f"Độ chính xác (Accuracy): {accuracy_score(y_test_fixed, y_pred):.4f}")
print("\nBáo cáo chi tiết (Classification Report):")
print(classification_report(y_test_fixed, y_pred))

Đang huấn luyện mô hình cơ sở trên 54 ứng dụng...

--- KẾT QUẢ HOÀN THÀNH WEEK 3 ---
Độ chính xác (Accuracy): 0.7318

Báo cáo chi tiết (Classification Report):
              precision    recall  f1-score   support

           0       0.73      0.46      0.57       496
           1       0.78      0.39      0.52        54
           2       0.00      0.00      0.00         5
           3       0.00      0.00      0.00        11
           5       0.71      0.37      0.48        82
           6       0.82      0.68      0.74        40
           7       0.00      0.00      0.00         1
           8       0.67      0.50      0.57         4
           9       0.96      0.69      0.81       157
          10       1.00      0.17      0.29         6
          11       0.00      0.00      0.00         7
          13       0.84      0.75      0.79       164
          15       0.00      0.00      0.00         2
          16       0.44      0.07      0.12       222
          17       0.71      

c:\Users\ADMIN\OneDrive\Máy tính\TTCS\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\ADMIN\OneDrive\Máy tính\TTCS\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\ADMIN\OneDrive\Máy tính\TTCS\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capital

In [19]:
from imblearn.over_sampling import SMOTE
from collections import Counter
import pandas as pd
import numpy as np

# 1. Lọc bỏ các class quá ít mẫu như cũ
counts = Counter(y_train_fixed)
valid_classes = [cls for cls, count in counts.items() if count >= 6]
mask = np.isin(y_train_fixed, valid_classes)
X_train_filtered = X_train_scaled[mask]
y_train_filtered = y_train_fixed[mask]

# 2. THIẾT LẬP CHIẾN THUẬT SMOTE THÔNG MINH
current_counts = Counter(y_train_filtered)
# có thể thay con số 2000 này tùy ý (ví dụ 1000, 3000 hoặc 5000)
# Đây là mức "vừa đủ" để AI không bỏ quên app nghèo mà không làm loãng app giàu
target_threshold = 2000 

# Tạo danh sách yêu cầu cho SMOTE
# Nếu app có < 2000 mẫu -> Bơm lên 2000
# Nếu app có > 2000 mẫu -> Giữ nguyên số lượng gốc
sampling_strategy = {
    label: max(count, target_threshold) 
    for label, count in current_counts.items()
}

smote = SMOTE(sampling_strategy=sampling_strategy, k_neighbors=3, random_state=42)

# 3. Chạy SMOTE
print(f"🚀 Đang SMOTE với ngưỡng tối thiểu {target_threshold} mẫu mỗi app...")
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_filtered, y_train_filtered)

# 4. Kiểm tra
new_counts = Counter(y_train_resampled)
print(f"Số lượng mẫu SAU khi SMOTE (đã cân bằng thông minh): {new_counts}")
print(f"Tổng số dòng dữ liệu mới: {len(X_train_resampled)}")

🚀 Đang SMOTE với ngưỡng tối thiểu 2000 mẫu mỗi app...
Số lượng mẫu SAU khi SMOTE (đã cân bằng thông minh): Counter({np.int64(17): 21488, np.int64(20): 15419, np.int64(23): 13861, np.int64(39): 8976, np.int64(21): 7222, np.int64(53): 3803, np.int64(51): 2000, np.int64(13): 2000, np.int64(32): 2000, np.int64(26): 2000, np.int64(0): 2000, np.int64(45): 2000, np.int64(5): 2000, np.int64(29): 2000, np.int64(46): 2000, np.int64(16): 2000, np.int64(28): 2000, np.int64(52): 2000, np.int64(1): 2000, np.int64(9): 2000, np.int64(40): 2000, np.int64(6): 2000, np.int64(36): 2000, np.int64(3): 2000, np.int64(11): 2000, np.int64(8): 2000, np.int64(50): 2000, np.int64(10): 2000, np.int64(24): 2000, np.int64(37): 2000, np.int64(25): 2000, np.int64(18): 2000, np.int64(2): 2000, np.int64(30): 2000, np.int64(48): 2000, np.int64(22): 2000, np.int64(41): 2000, np.int64(15): 2000, np.int64(44): 2000})
Tổng số dòng dữ liệu mới: 136769


In [20]:
#ô code 6
import tensorflow as tf
from tensorflow.keras.models import Sequential          
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Dropout
from tensorflow.keras.layers import BatchNormalization

In [21]:
# ô code 7 - ĐÃ SỬA ĐỂ DÙNG DỮ LIỆU SMOTE
import numpy as np

# dùng X_train_resampled (dữ liệu sau SMOTE)
X_train_cnn = np.expand_dims(X_train_resampled, axis=2)

# X_test thì vẫn giữ nguyên X_test_fixed (vì tập test không được SMOTE)
X_test_cnn = np.expand_dims(X_test_fixed, axis=2)

print(f"✅ Cấu trúc dữ liệu 3D đã sẵn sàng (Đã bao gồm SMOTE): {X_train_cnn.shape}")

✅ Cấu trúc dữ liệu 3D đã sẵn sàng (Đã bao gồm SMOTE): (136769, 80, 1)


In [22]:
# ô code 8
num_features = X_train_scaled.shape[1] # Tự động lấy số lượng cột (đặc trưng)
num_classes = len(le_final.classes_)   # Tự động lấy số lượng ứng dụng

model = Sequential([
    # Lớp lọc đặc trưng 1(Convolutional Layer)
    Conv1D(64, 3, activation='relu', input_shape=(num_features, 1)),
    BatchNormalization(), #chuẩn hóa theo lô (giúp dữ liệu ổn định hơn)
    MaxPooling1D(2),
    #Lớp lọc đặc trưng 2 
    Conv1D(128, 3, activation='relu', input_shape=(num_features, 1)),
    BatchNormalization(),
    MaxPooling1D(2),
    Dropout(0.3), #giảm 30% neuron để tránh học vẹt 
    # Lớp làm phẳng dữ liệu để đưa vào mạng nơ-ron
    Flatten(),
    Dense(256, activation='relu'), #256 neuron
    Dropout(0.5), # drop thêm neuron để dữ liệu chạy ổn định 
    # Lớp đầu ra (Số lượng app thực tế)
    Dense(num_classes, activation='softmax') # Khớp với số lượng app thực tế
])
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()

c:\Users\ADMIN\OneDrive\Máy tính\TTCS\.venv\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d_6 (Conv1D)               │ (None, 78, 64)         │           256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_6           │ (None, 78, 64)         │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_6 (MaxPooling1D)  │ (None, 39, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_7 (Conv1D)               │ (None, 37, 128)        │        24,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_7           │ (None, 37, 128)        │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_7 (MaxPooling1D)  │ (None, 18, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_6 (Dropout)             │ (None, 18, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_3 (Flatten)             │ (None, 2304)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 256)            │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_7 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 54)             │        13,878 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 629,686 (2.40 MB)

 Trainable params: 629,302 (2.40 MB)

 Non-trainable params: 384 (1.50 KB)

In [ ]:
# ô code 9
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# 1. Tự động dừng nếu 5 vòng không tăng accuracy (tiết kiệm thời gian cho Mạnh)
early_stop = EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True) 

# 2. Tự động giảm tốc độ học nếu bị "kẹt" (giúp mô hình học sâu hơn)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=3)
# Sử dụng y_train_fixed và y_test_fixed để khớp với mã hóa của XGBoost ở trên
history = model.fit(
    X_train_cnn, 
    y_train_resampled, 
    epochs=50, 
    batch_size=128, 
    validation_data=(X_test_cnn, y_test_fixed),
    
    callbacks=[
        tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5),
        reduce_lr 
    ]
)

Epoch 1/20
1069/1069 ━━━━━━━━━━━━━━━━━━━━ 16s 14ms/step - accuracy: 0.3811 - loss: 2.2692 - val_accuracy: 0.5247 - val_loss: 1.5662 - learning_rate: 0.0010
Epoch 2/20
1069/1069 ━━━━━━━━━━━━━━━━━━━━ 15s 14ms/step - accuracy: 0.4806 - loss: 1.7944 - val_accuracy: 0.5551 - val_loss: 1.4935 - learning_rate: 0.0010
Epoch 3/20
1069/1069 ━━━━━━━━━━━━━━━━━━━━ 15s 14ms/step - accuracy: 0.5192 - loss: 1.6290 - val_accuracy: 0.5623 - val_loss: 1.4330 - learning_rate: 0.0010
Epoch 4/20
1069/1069 ━━━━━━━━━━━━━━━━━━━━ 15s 14ms/step - accuracy: 0.5401 - loss: 1.5353 - val_accuracy: 0.5722 - val_loss: 1.3525 - learning_rate: 0.0010
Epoch 5/20
1069/1069 ━━━━━━━━━━━━━━━━━━━━ 15s 14ms/step - accuracy: 0.5568 - loss: 1.4705 - val_accuracy: 0.5742 - val_loss: 1.3625 - learning_rate: 0.0010
Epoch 6/20
1069/1069 ━━━━━━━━━━━━━━━━━━━━ 15s 14ms/step - accuracy: 0.5692 - loss: 1.4157 - val_accuracy: 0.5938 - val_loss: 1.3229 - learning_rate: 0.0010
Epoch 7/20
1069/1069 ━━━━━━━━━━━━━━━━━━━━ 15s 14ms/step - accura

In [24]:
#ô code 10 
from sklearn.metrics import classification_report
import numpy as np

# Dự đoán trên tập test
y_pred = model.predict(X_test_cnn) #cho mô hình làm bài kiểm tra trên tập test
y_pred_classes = np.argmax(y_pred, axis=1) # chọn ra trong 54 con số xác suất thì ra con số lớn nhất 

## Tìm danh sách các mã số (ID) thực sự xuất hiện trong tập Test
labels_in_test = np.unique(y_test_fixed)

#Lấy tên ứng dụng và ÉP KIỂU SANG STRING 
target_names_in_test = [str(le_final.classes_[i]) for i in labels_in_test]

# In báo cáo chi tiết cho 54 ứng dụng
print(classification_report(y_test_fixed, y_pred_classes, labels=labels_in_test, target_names=target_names_in_test)) # so sánh kết quả kiểm tra với bộ test ban đầu sau đó in ra các nhãn theo tên luôn chứ ko phải alf số 1, 2, 3, ...

625/625 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
              precision    recall  f1-score   support

           0       0.82      0.28      0.42       496
           1       0.36      0.50      0.42        54
           2       0.02      0.20      0.03         5
           3       0.05      0.18      0.08        11
           5       0.41      0.35      0.38        82
           6       0.85      0.70      0.77        40
           7       0.00      0.00      0.00         1
           8       0.50      1.00      0.67         4
           9       0.79      0.68      0.73       157
          10       0.38      0.50      0.43         6
          11       0.02      0.14      0.03         7
          13       0.69      0.72      0.70       164
          15       0.33      0.50      0.40         2
          16       0.46      0.05      0.09       222
          17       0.58      0.75      0.65      5391
          18       0.00      0.00      0.00         8
          20       0.76      0.79      0

c:\Users\ADMIN\OneDrive\Máy tính\TTCS\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\ADMIN\OneDrive\Máy tính\TTCS\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\ADMIN\OneDrive\Máy tính\TTCS\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capital